---
title: "DRG Cleaning"

author: "Carlos Resurreccion"

date: "2024-07-01"
---

In [10]:
knitr::opts_chunk$set(echo = TRUE)

## Load Required Libraries

In [11]:
options(verbose = FALSE)
options(warn = -1)

In [12]:
suppressPackageStartupMessages({
  # library(tidyverse)
  library(data.table)
  library(here)
  library(tictoc)
  library(stringr)
  library(stringi)
  library(lubridate)
  library(docstring)
  library(profvis)
  library(hash)
  # library(foreach)
  # library(doParallel)
  # library(parallel)
  library(future)
  library(future.apply)
  library(knitr)
})
print("Packages loaded successfully.")
here::here()

[1] "Packages loaded successfully."
[1] "Packages loaded successfully."


[1] "C:/"

In [13]:
options(warn = 1)

## Set Parameters

### Year to Load & Version

In [14]:
year_to_load <- "2018"
version <- "v2"

### Parameters

In [15]:
sample_size <- 25 * 1e3
seed <- 123

drop_cols <- c(paste0("ICDCODE", 13:14), "ICCODED15", paste0("ICDCODE", 16:170))
icd_cols <- paste0("clin_icd", 1:12)
rvs_cols <- paste0("clin_rvs", 1:20)


In [16]:
to_read <- FALSE
to_sample <- TRUE
to_write <- TRUE
to_group <- TRUE
to_filter <- FALSE # unused
to_profvis <- FALSE
to_chunk <- FALSE
to_view_checks <- TRUE


In [17]:
set.seed(seed)

options(future.globals.maxSize = 1024 * 1024 ^ 2)

global_seed <- seed #for parallelized operations


## Source Data Formats

In [18]:
source(here("data-cleaning", "r_scripts", "data-formats.R"))
source(here("data-cleaning", "r_scripts", "file-paths.R"))

Warning in file(filename, "r", encoding = encoding) :
  cannot open file 'C://data-cleaning/r_scripts/data-formats.R': No such file or directory


: [1m[33mError[39m in `file()`:[22m
[33m![39m cannot open the connection

## Source Functions

In [ ]:
source(here("data-cleaning", "r_scripts", "general-functions.R"))
source(here("data-cleaning", "r_scripts", "main-functions.R"))
source(here("data-cleaning", "r_scripts", "icd-functions.R"))
source(here("data-cleaning", "r_scripts", "rvs-functions.R"))
source(here("data-cleaning", "r_scripts", "pdx-functions.R"))
source(here("data-cleaning", "r_scripts", "grouper-functions.R"))

## Load Mapping Data

In [ ]:
proc <- fread(here(path_to_excel, "proc.csv"))
proc[, CODE := as.character(CODE)]

rvs_icd9 <- fread(here(path_to_aux, "rvs_icd9cm.csv"),
                  select = c("rvs", "icd9cm"))
rvs_icd9[, rvs := as.character(rvs)]
rvs_icd9[, icd9cm := as.character(icd9cm * 100)]
rvs_icd9 <- merge(rvs_icd9, proc[, .(CODE, DRGUSE)],
                  by.x = "icd9cm", by.y = "CODE", all.x = TRUE)
rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE]
rvs_icd9 <- rvs_icd9[!is.na(rvs) & !is.na(icd9cm), -"DRGUSE"]

acr_rvs <- fread(here(path_to_aux, "acr_rvs.csv"))

# Read in the data.table
tdrg_icd10 <- fread(here(path_to_aux, "i10.csv"))

# Set the key if not already set
setkey(tdrg_icd10, "CODE")

# Subset and assign the result to acc_pdx
acc_pdx <- tdrg_icd10[ACCPDX == "Y", CODE]

# Optional: if CODEs are not unique in tdrg_icd10
acc_pdx <- unique(acc_pdx)